# MedCompress — Pruning & Sparse Attention
**Author: Abhishek Shekhar**

Structured filter pruning and sparse attention ablation on ISIC 2020.

**Part A:** Structured pruning at sparsity 30/50/70%
**Part B:** Sparse bottleneck attention ablation

> All experiments auto-resume from Drive checkpoints.

In [ ]:
import os, sys, time, random, json, warnings
warnings.filterwarnings("ignore")
import numpy as np

# ── Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── GPU check ───────────────────────────────────────────────────────────────
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("No GPU! Go to Runtime → Change runtime type → T4 GPU")
tf.config.set_visible_devices(gpus[0], 'GPU')
tf.config.experimental.set_memory_growth(gpus[0], True)
print(f"TensorFlow {tf.__version__}  GPU: {gpus[0].name}")

# ── Directories (all outputs persist on Drive) ──────────────────────────────
EXP_NAME    = "pruning_sparse"   # ← overridden per notebook
DRIVE_BASE  = f"/content/drive/MyDrive/medcompress/pruning_sparse"
DATA_DIR    = f"/content/data/pruning_sparse"
RESULTS_DIR = f"{DRIVE_BASE}/results"
CKPT_DIR    = f"{DRIVE_BASE}/checkpoints"
MODELS_DIR  = f"{DRIVE_BASE}/models"
TFLITE_DIR  = f"{DRIVE_BASE}/tflite"
for d in [DRIVE_BASE, DATA_DIR, RESULTS_DIR, CKPT_DIR, MODELS_DIR, TFLITE_DIR]:
    os.makedirs(d, exist_ok=True)

SESSION_START = time.time()
print(f"Drive base : {DRIVE_BASE}")
print("Setup complete ✓")

In [ ]:
%%capture install_out
!pip install -q tensorflow-model-optimization tf2onnx onnx scikit-learn pillow tqdm
import tensorflow_model_optimization as tfmot
import tf2onnx
print("All packages installed ✓")

In [ ]:
# ── Kaggle credentials (run once per session) ──────────────────────────────
import os
KAGGLE_JSON = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(KAGGLE_JSON):
    from google.colab import files
    print("Upload your kaggle.json API token (from kaggle.com → Settings → API):")
    files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    print("Kaggle credentials saved ✓")
else:
    print("Kaggle credentials already present ✓")

In [ ]:
# ── Data (ISIC 2020) ────────────────────────────────────────────────────────
ISIC_DATA=f"/content/data/pruning_sparse/isic2020"; os.makedirs(ISIC_DATA,exist_ok=True)
if not os.path.exists(f"{ISIC_DATA}/train.csv"):
    !kaggle competitions download -c siim-isic-melanoma-classification -p {ISIC_DATA}
    !cd {ISIC_DATA} && unzip -q "*.zip" && rm -f *.zip
import pandas as pd
df=pd.read_csv(f"{ISIC_DATA}/train.csv")
jpeg_dir=f"{ISIC_DATA}/jpeg/train"
if not os.path.exists(jpeg_dir): jpeg_dir=f"{ISIC_DATA}/train"
df["image_path"]=df["image_name"].apply(lambda x: f"{jpeg_dir}/{x}.jpg")
df=df[df["image_path"].apply(os.path.exists)].reset_index(drop=True)
counts=df["target"].value_counts(); CLASS_WEIGHTS={0:len(df)/(2*counts[0]),1:len(df)/(2*counts[1])}
print(f"Dataset: {len(df)} samples")

In [ ]:
IMG_SIZE=224; BATCH_SIZE=32; SEEDS=[42,123,456,789,1024]
from tensorflow import keras; from tensorflow.keras import layers
import tensorflow_model_optimization as tfmot
from sklearn.model_selection import train_test_split

train_df,test_df=train_test_split(df,test_size=0.15,stratify=df["target"],random_state=42)
train_df,val_df =train_test_split(train_df,test_size=0.15/0.85,stratify=train_df["target"],random_state=42)

def make_ds(dataframe,augment=False,shuffle=False):
    paths=dataframe["image_path"].values; labels=dataframe["target"].values.astype(np.float32)
    def load(p,l):
        img=tf.image.decode_jpeg(tf.io.read_file(p),channels=3)
        img=tf.image.resize(img,[IMG_SIZE,IMG_SIZE])
        return tf.cast(img,tf.float32)/127.5-1.0, l
    def aug(img,l):
        img=tf.image.random_flip_left_right(img); img=tf.image.random_brightness(img,0.2); return img,l
    ds=tf.data.Dataset.from_tensor_slices((paths,labels))
    if shuffle: ds=ds.shuffle(len(paths),reshuffle_each_iteration=True)
    ds=ds.map(load,num_parallel_calls=tf.data.AUTOTUNE)
    if augment: ds=ds.map(aug,num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_ds(train_df,True,True); val_ds=make_ds(val_df); test_ds=make_ds(test_df)

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

def compute_cls_metrics(y_true, y_prob, thr=0.5):
    y_pred = (y_prob.ravel() >= thr).astype(int)
    y_true = y_true.astype(int).ravel()
    auc  = roc_auc_score(y_true, y_prob.ravel())
    f1   = f1_score(y_true, y_pred, zero_division=0)
    sens = recall_score(y_true, y_pred, zero_division=0)
    tn,fp,fn,tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    spec = tn/(tn+fp) if (tn+fp)>0 else 0.0
    return {"auc":float(auc),"f1":float(f1),"sensitivity":float(sens),"specificity":float(spec)}

def export_tflite(model, path, quant="fp32", calib_ds=None, n_calib=200):
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quant == "int8":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        if calib_ds:
            samples = []
            for imgs, _ in calib_ds:
                for i in range(len(imgs)):
                    samples.append(imgs[i].numpy())
                    if len(samples) >= n_calib: break
                if len(samples) >= n_calib: break
            def rep():
                for s in samples: yield [np.expand_dims(s,0)]
            conv.representative_dataset = rep
            conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
            conv.inference_input_type  = tf.uint8
            conv.inference_output_type = tf.uint8
    elif quant == "fp16":
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
        conv.target_spec.supported_types = [tf.float16]
    data = conv.convert()
    with open(path,"wb") as f: f.write(data)
    mb = os.path.getsize(path)/1e6
    print(f"  TFLite {quant}: {os.path.basename(path)} ({mb:.1f} MB)")
    return mb

def eval_tflite(path, dataset, warmup=5):
    interp = tf.lite.Interpreter(model_path=path, num_threads=4)
    interp.allocate_tensors()
    inp_d = interp.get_input_details()[0]
    out_d = interp.get_output_details()[0]
    preds, labels, lats = [], [], []
    for imgs, lbls in dataset:
        for i in range(len(imgs)):
            x = np.expand_dims(imgs[i].numpy(), 0)
            if inp_d["dtype"] == np.uint8:
                sc,zp = inp_d["quantization"]; x=(x/sc+zp).astype(np.uint8)
            interp.set_tensor(inp_d["index"], x)
            t0=time.perf_counter(); interp.invoke(); t1=time.perf_counter()
            lats.append((t1-t0)*1000)
            r=interp.get_tensor(out_d["index"])
            if out_d["dtype"]==np.uint8:
                sc,zp=out_d["quantization"]; r=(r.astype(np.float32)-zp)*sc
            preds.append(r.squeeze()); labels.append(lbls[i].numpy())
    lats = np.array(lats[warmup:])
    m = compute_cls_metrics(np.array(labels), np.array(preds))
    m["latency_ms"] = float(np.median(lats))
    m["latency_p95_ms"] = float(np.percentile(lats,95))
    m["size_mb"] = os.path.getsize(path)/1e6
    print(f"  TFLite AUC={m['auc']:.4f}  lat={m['latency_ms']:.1f}ms  sz={m['size_mb']:.1f}MB")
    return m

def done_flag(name): return f"{RESULTS_DIR}/{name}.done"
def is_done(name): return os.path.exists(done_flag(name))
def mark_done(name, result):
    with open(f"{RESULTS_DIR}/{name}.json","w") as f: json.dump(result, f)
    open(done_flag(name),"w").close()
def load_result(name):
    with open(f"{RESULTS_DIR}/{name}.json") as f: return json.load(f)

def train_model(model, train_ds, val_ds, cfg, name, monitor="val_auc", mode="max"):
    """Train with Drive checkpoint + resume. Returns history."""
    ckpt = f"{CKPT_DIR}/{name}_best.keras"
    epoch_f = f"{CKPT_DIR}/{name}_epoch.json"
    initial_epoch = 0
    if os.path.exists(ckpt):
        try:
            model.load_weights(ckpt)
            if os.path.exists(epoch_f):
                initial_epoch = json.load(open(epoch_f)).get("epoch",0)
            print(f"  Resumed from epoch {initial_epoch} ✓")
        except Exception as e:
            print(f"  Checkpoint load failed ({e}), starting fresh")
    class _EpochLog(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            json.dump({"epoch":epoch+1}, open(epoch_f,"w"))
    cbs = [
        tf.keras.callbacks.ModelCheckpoint(
            ckpt, save_best_only=True, monitor=monitor, mode=mode, verbose=0),
        tf.keras.callbacks.EarlyStopping(
            patience=cfg.get("patience",7), monitor=monitor, mode=mode,
            restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=3, mode=mode, verbose=0),
        _EpochLog()
    ]
    history = model.fit(
        train_ds, validation_data=val_ds,
        epochs=cfg.get("epochs",20), initial_epoch=initial_epoch,
        class_weight=cfg.get("class_weight"), callbacks=cbs, verbose=1)
    return model, history

print("Utilities loaded ✓")

In [ ]:
# ── Base model (EfficientNetB0) ───────────────────────────────────────────────
def build_base():
    bb=keras.applications.EfficientNetB0(include_top=False,weights="imagenet",input_shape=(224,224,3))
    for l in bb.layers[:-20]: l.trainable=False
    inp=keras.Input((224,224,3)); x=bb(inp,training=False)
    x=layers.GlobalAveragePooling2D()(x); x=layers.BatchNormalization()(x)
    x=layers.Dropout(0.3)(x); x=layers.Dense(256,activation="relu")(x)
    x=layers.Dropout(0.15)(x)
    return keras.Model(inp,layers.Dense(1,activation="sigmoid")(x),name="effb0_base")

# Load or train baseline
teacher_path=f"/content/drive/MyDrive/medcompress/isic_classification/models/efficientnetb0_s42.keras"
if os.path.exists(teacher_path):
    base_model=keras.models.load_model(teacher_path); print("Loaded from ISIC notebook ✓")
else:
    print("Training base EfficientNetB0...")
    base_model=build_base()
    base_model.compile(keras.optimizers.Adam(1e-4),"binary_crossentropy",[keras.metrics.AUC(name="auc")])
    base_model.fit(train_ds,validation_data=val_ds,epochs=20,class_weight=CLASS_WEIGHTS,verbose=1)
    base_model.save(f"{MODELS_DIR}/base_effb0.keras")

ref_metrics=eval_model(base_model,test_ds,"Base EfficientNetB0")
print(f"Baseline AUC: {ref_metrics['auc']:.4f}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PART A — Structured Magnitude Pruning (30% / 50% / 70%)
# ══════════════════════════════════════════════════════════════════════════════
SPARSITIES=[0.30, 0.50, 0.70]
pruning_results=[]

for sparsity in SPARSITIES:
    for seed in SEEDS:
        key=f"prune_{int(sparsity*100)}_s{seed}"
        if is_done(key):
            r=load_result(key); pruning_results.append(r)
            print(f"⏩ {key}  AUC={r['auc']:.4f}  size={r.get('size_mb',0):.1f}MB"); continue
        print(f"\n── Pruning {int(sparsity*100)}% seed={seed} ──")
        set_seed(seed)
        # Apply magnitude pruning using tfmot
        def apply_pruning(layer):
            if isinstance(layer, keras.layers.Dense) and layer.units > 1:
                return tfmot.sparsity.keras.prune_low_magnitude(
                    layer,
                    pruning_schedule=tfmot.sparsity.keras.ConstantSparsity(
                        sparsity, begin_step=0, frequency=100))
            return layer
        pruned_model = keras.models.clone_model(
            base_model, clone_function=apply_pruning)
        pruned_model.set_weights(base_model.get_weights())
        pruned_model.compile(keras.optimizers.Adam(1e-5),"binary_crossentropy",
                              [keras.metrics.AUC(name="auc")])
        # Fine-tune with pruning callbacks
        callbacks=[
            tfmot.sparsity.keras.UpdatePruningStep(),
            tfmot.sparsity.keras.PruningSummaries(log_dir=f"{CKPT_DIR}/prune_logs"),
            keras.callbacks.EarlyStopping(monitor="val_auc",patience=5,mode="max",
                                          restore_best_weights=True),
        ]
        pruned_model.fit(train_ds,validation_data=val_ds,epochs=10,
                         class_weight=CLASS_WEIGHTS,callbacks=callbacks,verbose=1)
        # Strip pruning + export
        stripped=tfmot.sparsity.keras.strip_pruning(pruned_model)
        p=f"{TFLITE_DIR}/pruned_{int(sparsity*100)}_s{seed}.tflite"
        conv=tf.lite.TFLiteConverter.from_keras_model(stripped)
        conv.optimizations=[tf.lite.Optimize.DEFAULT]
        with open(p,"wb") as f: f.write(conv.convert())
        r=eval_model(stripped,test_ds,key)
        r["sparsity"]=sparsity; r["size_mb"]=os.path.getsize(p)/1e6; r["seed"]=seed
        mark_done(key,r); pruning_results.append(r)
        print(f"  AUC={r['auc']:.4f}  Size={r['size_mb']:.1f}MB")

prune_df=pd.DataFrame(pruning_results)
print("\nPruning Results:")
print(prune_df.groupby("sparsity")[["auc","size_mb"]].mean().round(4))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PART B — Sparse Bottleneck Attention Ablation
# ══════════════════════════════════════════════════════════════════════════════
# Compare: base | + channel attention | + spatial attention | + both (CBAM-lite)

class ChannelAttention(layers.Layer):
    def __init__(self, ratio=8, **kw): super().__init__(**kw); self.ratio=ratio
    def build(self, shape):
        C=shape[-1]
        self.w1=self.add_weight("w1",(C//self.ratio,C),initializer="glorot_uniform",trainable=True)
        self.w2=self.add_weight("w2",(C,C//self.ratio),initializer="glorot_uniform",trainable=True)
    def call(self,x):
        avg=tf.reduce_mean(x,[1,2],keepdims=False); mx=tf.reduce_max(x,[1,2],keepdims=False)
        def exc(v): return tf.matmul(tf.nn.relu(tf.matmul(v,self.w2)),self.w1)
        return x*tf.nn.sigmoid(exc(avg)+exc(mx))[:,tf.newaxis,tf.newaxis,:]

class SpatialAttention(layers.Layer):
    def __init__(self, **kw):
        super().__init__(**kw)
        self.conv=layers.Conv2D(1,7,padding="same",activation="sigmoid",use_bias=False)
    def call(self,x):
        avg=tf.reduce_mean(x,-1,keepdims=True); mx=tf.reduce_max(x,-1,keepdims=True)
        return x*self.conv(tf.concat([avg,mx],-1))

def build_with_attention(attn_type="both"):
    bb=keras.applications.EfficientNetB0(include_top=False,weights="imagenet",input_shape=(224,224,3))
    for l in bb.layers[:-20]: l.trainable=False
    inp=keras.Input((224,224,3)); x=bb(inp,training=False)
    if attn_type in ("channel","both"): x=ChannelAttention(name="ca")(x)
    if attn_type in ("spatial","both"): x=SpatialAttention(name="sa")(x)
    x=layers.GlobalAveragePooling2D()(x); x=layers.BatchNormalization()(x)
    x=layers.Dropout(0.3)(x); x=layers.Dense(256,activation="relu")(x)
    x=layers.Dropout(0.15)(x)
    return keras.Model(inp,layers.Dense(1,activation="sigmoid")(x),name=f"effb0_{attn_type}")

attn_results=[]
for attn_type in ["none","channel","spatial","both"]:
    for seed in SEEDS[:3]:  # 3 seeds for ablation
        key=f"attn_{attn_type}_s{seed}"
        if is_done(key):
            r=load_result(key); attn_results.append(r)
            print(f"⏩ {key}  AUC={r['auc']:.4f}"); continue
        print(f"\n── Attention={attn_type} seed={seed} ──")
        set_seed(seed)
        m = base_model if attn_type=="none" else build_with_attention(attn_type)
        if attn_type!="none":
            m.compile(keras.optimizers.Adam(1e-4),"binary_crossentropy",[keras.metrics.AUC(name="auc")])
            m,_=train_model(m,train_ds,val_ds,{"epochs":15,"patience":5,"class_weight":CLASS_WEIGHTS},
                            key,"val_auc")
        r=eval_model(m,test_ds,key); r["attn_type"]=attn_type; r["seed"]=seed
        mark_done(key,r); attn_results.append(r)

attn_df=pd.DataFrame(attn_results)
print("\nAttention Ablation:")
print(attn_df.groupby("attn_type")["auc"].agg(["mean","std"]).round(4))

# Save all pruning results
prune_df.to_csv(f"{RESULTS_DIR}/pruning_results.csv",index=False)
attn_df.to_csv(f"{RESULTS_DIR}/attention_ablation.csv",index=False)
print(f"\nAll saved to {RESULTS_DIR}")